# Data Cleaning
1. Dropping any rows with lsoa_code missing in street_crimes
2. Checking in lsoa_info if any lsoa code is missing from official list

In [3]:
import geopandas as gpd
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/police_data.db')
cursor = conn.cursor()

In [11]:
total_rows = pd.read_sql("SELECT COUNT(*) as total FROM street_crimes;", conn).iloc[0,0]
print(f'total rows {total_rows}')

null_query = """
SELECT 
    COUNT(*) - COUNT(crime_id) as missing_crime_id,
    COUNT(*) - COUNT(month) as missing_month,
    COUNT(*) - COUNT(longitude) as missing_long,
    COUNT(*) - COUNT(latitude) as missing_lat,
    COUNT(*) - COUNT(lsoa_code) as missing_lsoa,
    COUNT(*) - COUNT(crime_type) as missing_crime,
    COUNT(*) - COUNT(last_outcome) as missing_outcome
FROM street_crimes;
"""
null_df = pd.read_sql(null_query, conn)
null_percentages = (null_df / total_rows) * 100
display(null_percentages.round(2).astype(str) + '%')

crime_type_query = """
SELECT crime_type, COUNT(*) as incident_count 
FROM street_crimes 
GROUP BY crime_type 
ORDER BY incident_count DESC;
"""
crime_df = pd.read_sql(crime_type_query, conn)
crime_df['percentage'] = ((crime_df['incident_count'] / total_rows) * 100).round(2)
display(crime_df)

total rows 94019401


,missing_crime_id,missing_month,missing_long,missing_lat,missing_lsoa,missing_crime,missing_outcome
0,31.07%,0.0%,1.49%,1.49%,3.92%,0.0%,33.89%


,crime_type,incident_count,percentage
0,Anti-social behaviour,25048350,26.64
1,Violence and sexual offences,21460754,22.83
2,Criminal damage and arson,7567187,8.05
3,Other theft,7286898,7.75
4,Vehicle crime,5807870,6.18
5,Burglary,5505345,5.86
6,Shoplifting,5145980,5.47
7,Public order,4768815,5.07
8,Other crime,3124502,3.32
9,Drugs,2542571,2.70


In [21]:
#1 dropping all rows that have no lsoa_code

#count the rows before deletion
cursor.execute('SELECT COUNT(*) FROM street_crimes;')
count_before = cursor.fetchone()
print(f'count before deletion {count_before}')

#deletes all rows where lsoa_code is empty
delete = 'DELETE FROM street_crimes WHERE lsoa_code IS NULL;'
cursor.execute(delete)
conn.commit()

#count the rows after deletion
cursor.execute("SELECT COUNT(*) FROM street_crimes;")
count_after = cursor.fetchone()
print(f'count after deletion {count_before}')

count before deletion (90337708,)
count after deletion (90337708,)


In [18]:
#2 check in lsoa_info 

#clean data gathered
clean_lsoas_df = pd.read_csv("../data/lsoa_codes.csv", usecols=['LSOA21CD', 'LSOA21NM'])
total_lsoa_set = set(clean_lsoas_df['LSOA21CD'])

#get from db all lsoas
db_lsoas_df = pd.read_sql("SELECT lsoa_code FROM lsoa_info;",  conn)
db_lsoa_set = set(db_lsoas_df['lsoa_code'])

#set difference to find which lsoa's missing from our db
missing_lsoas = total_lsoa_set - db_lsoa_set

print(f'total lsoa codes count (clean) {len(total_lsoa_set)}')
print(f'database lsoa codes count {len(db_lsoa_set)}')
print(f'number of missing lsoas {len(missing_lsoas)}')

#convert missing lsoas to series
missing_series = pd.Series(list(missing_lsoas))

#take the first letter in each code and count
code_missing = missing_series.str[0].value_counts()

print(f'missing lsoas come from {code_missing}')

total lsoa codes count (clean) 35672
database lsoa codes count 33755
number of missing lsoas 1917
missing lsoas come from W    1917
Name: count, dtype: int64
